# Dev notebook — scalability_bench.py

Notebook này **import thẳng từ `scalability_bench.py`** (không định nghĩa lại) để:
1. Xem dữ liệu tổng hợp (synthetic) trông như thế nào và kiểm tra `compute_iaqi_hour()` ra kết quả hợp lý trên đó.
2. Chạy thử 1 benchmark NHỎ, nhanh, để hiểu cơ chế đo (không phải số liệu chính thức cho báo cáo).
3. Đọc lại CSV kết quả chính thức (chạy bằng `run_scalability.sh`, mỗi combo 1 tiến trình riêng — xem docstring trong `scalability_bench.py` để biết lý do) và vẽ lại biểu đồ ngay trong notebook để xem trực tiếp.

**Lưu ý:** không chạy full lưới 100K/1M/8M trong notebook này — việc đó tốn nhiều phút và nên chạy qua `run_scalability.sh` (mỗi combo 1 tiến trình Python riêng, tránh lỗi ngẫu nhiên khi tạo nhiều SparkSession liên tiếp trong cùng 1 process — xem ghi chú đầu `scalability_bench.py`).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from scalability_bench import make_synthetic_df, run_once, load_csv, print_table, plot_charts, DEFAULT_CSV, DEFAULT_IMAGES
from phase2_aqi import compute_iaqi_hour
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("scalability_dev").master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Bước 1 — xem dữ liệu tổng hợp trông như thế nào

Cố tình chạy quy mô NHỎ (5.000 dòng) ở đây — chỉ để xem/hiểu, không phải đo tốc độ.

In [ ]:
df = make_synthetic_df(spark, n_rows=5000, n_stations=20)
print("so dong:", df.count(), "| so tram:", df.select("station_id").distinct().count())
df.orderBy("station_id", "ts_epoch").show(5)

## Bước 2 — dữ liệu giả có làm `compute_iaqi_hour()` ra kết quả hợp lý không?

Vì đây là dữ liệu random đều (không mô phỏng phân phối thật như `generate_sample.py`), AQI sẽ trải khá rộng — chỉ cần không có lỗi/toàn NULL là đạt yêu cầu cho mục đích benchmark tốc độ.

In [ ]:
result = compute_iaqi_hour(df)
result.select("aqi", "aqi_level", "aqi_label", "dominant_pollutant").summary().show()
print("so dong aqi null (ky vong 0 vi du lieu gia luon du 6 chat):", result.filter(F.col("aqi").isNull()).count())

## Bước 3 — chạy thử 1 benchmark nhỏ để hiểu cơ chế đo

`run_once()` tự tạo SparkSession riêng (`local[N]`) rồi `stop()` khi xong — ở đây gọi trực tiếp trong notebook (đang có sẵn 1 session `local[2]` cho các bước trên) nên sẽ thấy log tạo thêm 1 session nữa, đúng như cách `scalability_bench.py` chạy độc lập.

In [ ]:
secs = run_once(n_rows=20_000, n_executors=2)
print(f"20,000 dòng / 2 executor -> {secs:.2f}s (chỉ để minh hoạ, KHÔNG phải số chính thức)")

## Bước 4 — đọc kết quả CHÍNH THỨC (từ `run_scalability.sh`) và vẽ lại trong notebook

Cần chạy xong `./run_scalability.sh` trước (9 tổ hợp, có thể mất nhiều phút, đặc biệt ở 8M dòng).

In [ ]:
rows = load_csv(DEFAULT_CSV)
print_table(rows)

In [ ]:
plot_charts(rows, DEFAULT_IMAGES)

# Dung IPython.display thay vi plt.imshow/plt.show(): plot_charts() da goi
# matplotlib.use("Agg") de chay headless (can cho CLI/background) -> Agg backend
# KHONG hien duoc qua plt.show(), nhung hien anh tinh (da luu file) thi luon duoc,
# khong phu thuoc backend dang active.
from IPython.display import Image, display

display(Image(filename=f"{DEFAULT_IMAGES}/scalability_runtime.png"))
display(Image(filename=f"{DEFAULT_IMAGES}/scalability_speedup.png"))

## Ghi chú

- Số liệu chính thức cho báo cáo (`docs/experiments.md`) PHẢI lấy từ `run_scalability.sh`, không lấy từ Bước 3 (chỉ minh hoạ, dùng quy mô nhỏ, có thể lẫn với session `local[2]` đang mở sẵn trong notebook làm sai lệch số đo).
- Sửa `scalability_bench.py` xong, chạy lại cell là thấy ngay nhờ `%autoreload 2`.